In [11]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [12]:
from pathlib import Path

DOCS_DIR = Path("docs")

# Загрузка и чанкинг - это вы уже видели в теории
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 543


In [13]:
# 1. Сколько разных файлов дали чанки?
unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

Файлов с чанками: 61


In [14]:
# Counter из модуля collections умеет считать частоты
from collections import Counter

# 2. Сколько чанков пришло из каждого файла?
#    Подсказка: соберите Counter по c["source"] для всех чанков
source_counts = Counter(c["source"] for c in chunks)
for i in source_counts:
    print(f'{source_counts[i]}\t{i}')

19	docs\api\anthropic-compatibility.mdx
4	docs\api\authentication.mdx
3	docs\api\errors.mdx
6	docs\api\introduction.mdx
20	docs\api\openai-compatibility.mdx
4	docs\api\streaming.mdx
2	docs\api\usage.mdx
72	docs\api.md
7	docs\capabilities\embeddings.mdx
7	docs\capabilities\streaming.mdx
11	docs\capabilities\structured-outputs.mdx
8	docs\capabilities\thinking.mdx
33	docs\capabilities\tool-calling.mdx
5	docs\capabilities\vision.mdx
19	docs\capabilities\web-search.mdx
3	docs\cli.mdx
7	docs\cloud.mdx
3	docs\context-length.mdx
13	docs\development.md
8	docs\docker.mdx
4	docs\examples.md
37	docs\faq.mdx
14	docs\gpu.mdx
5	docs\import.mdx
4	docs\index.mdx
7	docs\integrations\chatgpt.mdx
14	docs\integrations\claude-code.mdx
6	docs\integrations\claude-desktop.mdx
4	docs\integrations\cline-cli.mdx
3	docs\integrations\cline.mdx
4	docs\integrations\codex.mdx
6	docs\integrations\copilot-cli.mdx
5	docs\integrations\deepseek-harness.mdx
4	docs\integrations\droid.mdx
2	docs\integrations\goose.mdx
4	docs\

In [15]:
# 3. Топ-3 файла по числу чанков
top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Топ-3 файла по числу чанков:
  docs\api.md: 72 чанков
  docs\faq.mdx: 37 чанков
  docs\capabilities\tool-calling.mdx: 33 чанков


In [16]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    # Ваш код здесь: посчитайте количество чанков и среднюю длину
    # Выведите результат в формате: max_chars=200 -> чанков: X, средняя длина: Y
    print(f'>>> max_chars = {max_chars}\t{len(chunks_sample)}')

    average_length = sum(len(s) for s in chunks_sample) / len(chunks_sample) 
    print(int(average_length))

>>> max_chars = 200	159
328
>>> max_chars = 800	72
742
>>> max_chars = 3000	30
1785


In [17]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=200)

print(chunks_sample[5])

## Generate a completion

```
POST /api/generate
```


In [19]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=800)

print(chunks_sample[5])

#### Structured outputs

Structured outputs are supported by providing a JSON schema in the `format` parameter. The model will generate a response that matches the schema. See the [structured outputs](#request-structured-outputs) example below.

#### JSON mode

Enable JSON mode by setting the `format` parameter to `json`. This will structure the response as a valid JSON object. See the JSON mode [example](#request-json-mode) below.

> [!IMPORTANT]
> It's important to instruct the model to use JSON in the `prompt`. Otherwise, the model may generate large amounts whitespace.

### Examples

#### Generate request (Streaming)

##### Request

```shell
curl http://localhost:11434/api/generate -d '{
  "model": "llama3.2",
  "prompt": "Why is the sky blue?"
}'
```

##### Response


In [20]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=3000)

print(chunks_sample[5])

```shell
curl -X POST http://localhost:11434/api/generate -H "Content-Type: application/json" -d '{
  "model": "llama3.1:8b",
  "prompt": "Ollama is 22 years old and is busy saving the world. Respond using JSON",
  "stream": false,
  "format": {
    "type": "object",
    "properties": {
      "age": {
        "type": "integer"
      },
      "available": {
        "type": "boolean"
      }
    },
    "required": [
      "age",
      "available"
    ]
  }
}'
```

##### Response

```json
{
  "model": "llama3.1:8b",
  "created_at": "2024-12-06T00:48:09.983619Z",
  "response": "{\n  \"age\": 22,\n  \"available\": true\n}",
  "done": true,
  "done_reason": "stop",
  "context": [1, 2, 3],
  "total_duration": 1075509083,
  "load_duration": 567678166,
  "prompt_eval_count": 28,
  "prompt_eval_duration": 236000000,
  "eval_count": 16,
  "eval_duration": 269000000
}
```

#### Request (JSON mode)

> [!IMPORTANT]
> When `format` is set to `json`, the output will always be a well-formed JSON object

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)' thrown while requesting HEAD https:/

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 4s [Retry 3/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 8s [Retry 4/5].
'[Wi